In [102]:
import sys, json, os, time, re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import seaborn as sns
from tqdm import tqdm
import random
import zlib
import importlib
from cycler import cycler
import types
import math

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.experiment_loader as experiment_loader
import multiomic_transformer.datasets.dataset_refactor as dataset_refactor
import multiomic_transformer.models.model as model_module
from multiomic_transformer.datasets.dataset_refactor import MultiChromosomeDataset
from multiomic_transformer.datasets.dataset_refactor import IndexedChromBucketBatchSampler

GROUND_TRUTH_DIR = PROJECT_DIR / "data" / "ground_truth_files"

In [103]:
def load_ground_truth(ground_truth_file):
    if type(ground_truth_file) == str:
        ground_truth_file = Path(ground_truth_file)
        
    if ground_truth_file.suffix == ".csv":
        sep = ","
    elif ground_truth_file.suffix == ".tsv":
        sep="\t"
        
    ground_truth_df = pd.read_csv(ground_truth_file, sep=sep, on_bad_lines="skip", engine="python")
    
    if "chip" in ground_truth_file.name and "atlas" in ground_truth_file.name:
        ground_truth_df = ground_truth_df[["source_id", "target_id"]]

    if ground_truth_df.columns[0] != "Source" or ground_truth_df.columns[1] != "Target":
        ground_truth_df = ground_truth_df.rename(columns={ground_truth_df.columns[0]: "Source", ground_truth_df.columns[1]: "Target"})
    ground_truth_df["Source"] = ground_truth_df["Source"].astype(str).str.upper()
    ground_truth_df["Target"] = ground_truth_df["Target"].astype(str).str.upper()
    
    # Build TF, TG, and edge sets for quick lookup later
    gt = ground_truth_df[["Source", "Target"]].dropna()

    gt_tfs = set(gt["Source"].unique())
    gt_tgs = set(gt["Target"].unique())
    
    gt_pairs = (gt["Source"] + "\t" + gt["Target"]).drop_duplicates()
    
    gt_lookup = (gt_tfs, gt_tgs, set(gt_pairs))
        
    return ground_truth_df, gt_lookup

gt_by_dataset_dict = {
    "mESC": {
        "ChIP-Atlas mESC": load_ground_truth(GROUND_TRUTH_DIR / "chip_atlas_tf_peak_tg_dist.csv"),
        "RN111": load_ground_truth(GROUND_TRUTH_DIR / "RN111.tsv"),
        "RN112": load_ground_truth(GROUND_TRUTH_DIR / "RN112.tsv"),
        "RN114": load_ground_truth(GROUND_TRUTH_DIR / "RN114.tsv"),
        "RN116": load_ground_truth(GROUND_TRUTH_DIR / "RN116.tsv"),        
    },
}

In [104]:
exp = experiment_loader.ExperimentLoader(
    experiment_dir = "/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments/",
    experiment_name="mESC_E7.5_rep1_muon_preprocessing",
    model_num=4,
)

In [111]:
import types
import math
import torch


def patched_scaled_dot_product_attention(self, Q, K, V, mask, attn_bias=None):
    Q = torch.nan_to_num(Q, nan=0.0, posinf=1e4, neginf=-1e4)
    K = torch.nan_to_num(K, nan=0.0, posinf=1e4, neginf=-1e4)
    V = torch.nan_to_num(V, nan=0.0, posinf=1e4, neginf=-1e4)

    scale = 1.0 / math.sqrt(self.d_k)
    logits = torch.matmul(Q.float(), K.float().transpose(-2, -1)) * scale

    if attn_bias is not None:
        attn_bias = torch.nan_to_num(attn_bias, nan=0.0, posinf=1e4, neginf=-1e4)
        logits = logits + attn_bias.float().clamp(-1e4, 1e4)

    probs = torch.softmax(logits, dim=-1)
    probs = torch.nan_to_num(probs, nan=0.0)

    self.last_logits = logits.detach()
    self.last_attn_probs = probs.detach()

    out = torch.matmul(probs, V.float()).to(Q.dtype)
    return out


def patch_loaded_model_in_place(model):
    # patch TF->ATAC attention
    model.cross_tf_to_atac.attn.last_logits = None
    model.cross_tf_to_atac.attn.last_attn_probs = None
    model.cross_tf_to_atac.attn.scaled_dot_product_attention = types.MethodType(
        patched_scaled_dot_product_attention,
        model.cross_tf_to_atac.attn
    )

    # patch TG->ATAC attention
    model.cross_tg_to_atac.attn.last_logits = None
    model.cross_tg_to_atac.attn.last_attn_probs = None
    model.cross_tg_to_atac.attn.scaled_dot_product_attention = types.MethodType(
        patched_scaled_dot_product_attention,
        model.cross_tg_to_atac.attn
    )

    return model

def run_tf_window_tg_attribution_single_gpu(
    selected_experiment_dir,
    model,
    test_loader,
    tg_scaler,
    tf_scaler,
    state,
    tf_names,
    tg_names,
    device,
    use_amp,
    max_batches: int = None,
    max_tgs_per_batch: int = None,
    top_tfs: int = 25,
    top_windows: int = 50,
    use_abs_score: bool = True,
):
    """
    Single-GPU, non-distributed TF-window-TG attribution.

    Saves:
        selected_experiment_dir / "tf_window_tg_attribution_raw.parquet"

    Returns:
        df_wide: DataFrame with TFs as rows and TGs as columns
    """
    T_total = len(tf_names)
    G_total = len(tg_names)

    score_sum = torch.zeros(T_total, G_total, device=device, dtype=torch.float32)
    score_count = torch.zeros_like(score_sum)

    model.to(device).eval()

    if max_batches is None:
        total_batches = len(test_loader)
    else:
        total_batches = min(max_batches, len(test_loader))

    iterator = tqdm(
        test_loader,
        desc="TF-window-TG attribution",
        unit="batches",
        total=total_batches,
        ncols=100,
    )

    for b_idx, batch in enumerate(iterator):
        if max_batches is not None and b_idx >= max_batches:
            break

        atac_wins, tf_tensor, targets, bias, tf_ids, tg_ids, motif_mask = batch
        atac_wins = atac_wins.to(device, non_blocking=True)
        tf_tensor = tf_tensor.to(device, non_blocking=True)
        bias = bias.to(device, non_blocking=True) if bias is not None else None
        tf_ids = tf_ids.to(device, non_blocking=True)
        tg_ids = tg_ids.to(device, non_blocking=True)
        motif_mask = motif_mask.to(device, non_blocking=True) if motif_mask is not None else None

        captured = {}

        def hook_gene_pred_dense(module, inputs, output):
            captured["tg_cross_attn_repr"] = inputs[0]

        handle = model.gene_pred_dense.register_forward_hook(hook_gene_pred_dense)

        try:
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                tf_scaled = (
                    tf_scaler.transform(tf_tensor, tf_ids)
                    if tf_scaler is not None
                    else tf_tensor
                )

                preds_s = model(
                    atac_wins,
                    tf_scaled,
                    tf_ids=tf_ids,
                    tg_ids=tg_ids,
                    bias=bias,
                    motif_mask=motif_mask,
                )

                if isinstance(preds_s, tuple):
                    preds_s = preds_s[0]

                preds_u = (
                    tg_scaler.inverse_transform(preds_s, tg_ids)
                    if tg_scaler is not None
                    else preds_s
                )
                preds_u = torch.nan_to_num(
                    preds_u.float(), nan=0.0, posinf=1e6, neginf=-1e6
                )
        finally:
            handle.remove()

        if "tg_cross_attn_repr" not in captured:
            raise RuntimeError("Failed to capture tg_cross_attn_repr from gene_pred_dense hook.")

        tg_cross_attn_repr = captured["tg_cross_attn_repr"]   # [B, G, D]
        tg_cross_attn_repr.retain_grad()

        tf_to_win = model.cross_tf_to_atac.attn.last_attn_probs.mean(dim=1)  # [B, T, W]
        tg_to_win = model.cross_tg_to_atac.attn.last_attn_probs.mean(dim=1)  # [B, G, W]

        B = preds_u.shape[0]
        G_eval = preds_u.shape[1]

        tg_indices = torch.arange(G_eval, device=device)
        chunk_size = max_tgs_per_batch if max_tgs_per_batch is not None else tg_indices.numel()

        for chunk_start in range(0, tg_indices.numel(), chunk_size):
            tg_chunk = tg_indices[chunk_start: chunk_start + chunk_size]

            for g in tg_chunk.tolist():
                model.zero_grad(set_to_none=True)
                if tg_cross_attn_repr.grad is not None:
                    tg_cross_attn_repr.grad.zero_()

                scalar = preds_u[:, g].sum()
                scalar.backward(retain_graph=True)

                for b in range(B):
                    grad_h = tg_cross_attn_repr.grad[b, g].detach().clone()  # [D]
                    tg_sensitivity = grad_h.abs().mean()

                    tfw = tf_to_win[b]         # [T, W]
                    tgw = tg_to_win[b, g]      # [W]
                    tfv = tf_tensor[b]         # [T]
                    winv = atac_wins[b, :, 0]  # [W]

                    tf_strength = (tfw * tfv.abs().unsqueeze(1)).max(dim=1).values
                    win_strength = (tgw * winv.abs()).abs()

                    top_tf_idx = torch.topk(
                        tf_strength,
                        k=min(top_tfs, tf_strength.numel())
                    ).indices

                    top_win_idx = torch.topk(
                        win_strength,
                        k=min(top_windows, win_strength.numel())
                    ).indices

                    tfw_sub = tfw[top_tf_idx][:, top_win_idx]   # [T', W']
                    tfv_sub = tfv[top_tf_idx].unsqueeze(1)      # [T', 1]
                    tgw_sub = tgw[top_win_idx].unsqueeze(0)     # [1, W']
                    winv_sub = winv[top_win_idx].unsqueeze(0)   # [1, W']

                    path_score = tfw_sub * tgw_sub
                    activity_score = path_score * tfv_sub * winv_sub
                    final_score = activity_score * tg_sensitivity

                    tf_tg_scores = final_score.sum(dim=1)  # [T']

                    if use_abs_score:
                        tf_tg_scores = tf_tg_scores.abs()

                    if tg_ids.dim() == 1:
                        tg_global = int(tg_ids[g].item())
                    else:
                        tg_global = int(tg_ids[b, g].item())

                    score_sum[:, tg_global].index_add_(0, top_tf_idx, tf_tg_scores)
                    score_count[:, tg_global].index_add_(0, top_tf_idx, torch.ones_like(tf_tg_scores))

        del (
            atac_wins,
            tf_tensor,
            bias,
            tf_ids,
            tg_ids,
            motif_mask,
            tf_scaled,
            preds_s,
            preds_u,
            tg_cross_attn_repr,
            tf_to_win,
            tg_to_win,
        )
        if device.type == "cuda":
            torch.cuda.empty_cache()

    tf_tg_attr = score_sum / (score_count + 1e-12)

    seen_tf_mask = score_count.sum(dim=1) > 0
    seen_tg_mask = score_count.sum(dim=0) > 0

    seen_tf_ids = torch.nonzero(seen_tf_mask, as_tuple=True)[0]
    seen_tg_ids = torch.nonzero(seen_tg_mask, as_tuple=True)[0]

    tf_tg_attr_compact = tf_tg_attr[seen_tf_mask][:, seen_tg_mask]
    tf_tg_attr_np = np.nan_to_num(tf_tg_attr_compact.detach().cpu().numpy(), nan=0.0)

    seen_tf_ids_np = seen_tf_ids.cpu().numpy()
    seen_tg_ids_np = seen_tg_ids.cpu().numpy()

    df_wide = pd.DataFrame(
        tf_tg_attr_np,
        index=[tf_names[i] for i in seen_tf_ids_np],
        columns=[tg_names[i] for i in seen_tg_ids_np],
    )

    print(
        f"TF-window-TG attribution: kept {len(seen_tf_ids_np)}/{T_total} TFs, "
        f"{len(seen_tg_ids_np)}/{G_total} TGs"
    )

    out_path = selected_experiment_dir / "tf_window_tg_attribution_raw.parquet"
    df_wide.to_parquet(out_path)
    print(f"Saved TF-window-TG attribution DataFrame to {out_path}")

    return df_wide


In [125]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

use_amp = True and device.type == "cuda"
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(device)
    if capability[0] < 7:
        print(
            f"GPU compute capability {capability[0]}.{capability[1]} < 7.0, disabling AMP"
        )
        use_amp = False

exp.load_trained_model("trained_model.pt")
exp.model = patch_loaded_model_in_place(exp.model)
exp.model.eval()

df_wide = run_tf_window_tg_attribution_single_gpu(
    selected_experiment_dir=exp.model_training_dir,
    model=exp.model,
    test_loader=exp.test_loader,
    tg_scaler=exp.tg_scaler,
    tf_scaler=exp.tf_scaler,
    state=exp.state,
    tf_names=exp.tf_names,
    tg_names=exp.tg_names,
    device=exp.device,
    use_amp=use_amp,
    max_batches=5,
    max_tgs_per_batch=None,
    top_tfs=25,
    top_windows=50,
    use_abs_score=True,
)

TF-window-TG attribution: 100%|██████████████████████████████████| 5/5 [02:20<00:00, 28.10s/batches]

TF-window-TG attribution: kept 505/718 TFs, 721/12004 TGs
Saved TF-window-TG attribution DataFrame to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments/mESC_E7.5_rep1_muon_preprocessing/model_training_004/tf_window_tg_attribution_raw.parquet


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
from typing import Optional


class TGOutputWrapperSingleContext(nn.Module):
    """
    Original-style wrapper for explaining one TG output at a time.
    Kept here for debugging / comparison with the faster all-output version.
    """
    def __init__(
        self,
        model: nn.Module,
        tf_ids: torch.Tensor,
        tg_ids: torch.Tensor,
        bias: Optional[torch.Tensor] = None,
        motif_mask: Optional[torch.Tensor] = None,
        target_tg_idx: int = 0,
        reduce: str = "single",
        context_index: int = 0,
    ):
        super().__init__()
        self.model = model
        self.target_tg_idx = int(target_tg_idx)
        self.reduce = reduce

        self.register_buffer("tf_ids_fixed", tf_ids.detach().clone())
        self.register_buffer("tg_ids_fixed", tg_ids.detach().clone())

        if bias is not None:
            if bias.dim() > 0 and bias.shape[0] > 1:
                self.register_buffer(
                    "bias_template",
                    bias[context_index:context_index + 1].detach().clone(),
                )
            else:
                self.register_buffer("bias_template", bias.detach().clone())
        else:
            self.bias_template = None

        if motif_mask is not None:
            if (
                bias is not None
                and motif_mask.dim() > 0
                and motif_mask.shape[0] > 1
                and motif_mask.shape[0] == bias.shape[0]
            ):
                self.register_buffer(
                    "motif_mask_template",
                    motif_mask[context_index:context_index + 1].detach().clone(),
                )
            else:
                self.register_buffer("motif_mask_template", motif_mask.detach().clone())
        else:
            self.motif_mask_template = None

    def _match_batch(self, x: Optional[torch.Tensor], batch_size: int):
        if x is None:
            return None
        if x.dim() == 1:
            return x
        if x.shape[0] == batch_size:
            return x
        if x.shape[0] == 1:
            reps = [batch_size] + [1] * (x.dim() - 1)
            return x.repeat(*reps)
        return x

    def forward(self, atac_wins: torch.Tensor, tf_tensor: Optional[torch.Tensor] = None):
        if tf_tensor is None:
            if isinstance(atac_wins, (list, tuple)) and len(atac_wins) == 2:
                atac_wins, tf_tensor = atac_wins
            else:
                raise ValueError(
                    "Expected forward(atac_wins, tf_tensor) or forward([atac_wins, tf_tensor])."
                )

        batch_size = atac_wins.shape[0]
        bias = self._match_batch(self.bias_template, batch_size)
        motif_mask = self._match_batch(self.motif_mask_template, batch_size)

        out, _, _ = self.model(
            atac_wins,
            tf_tensor,
            tf_ids=self.tf_ids_fixed,
            tg_ids=self.tg_ids_fixed,
            bias=bias,
            motif_mask=motif_mask,
            return_shortcut_contrib=False,
        )

        if self.reduce == "single":
            return out[:, self.target_tg_idx:self.target_tg_idx + 1]
        elif self.reduce == "sum":
            return out.sum(dim=1, keepdim=True)
        else:
            raise ValueError(f"Unsupported reduce mode: {self.reduce}")


class TGOutputWrapperAll(nn.Module):
    """
    Faster wrapper: returns all TG outputs at once so we can build one SHAP
    explainer per batch instead of one per target gene.
    """
    def __init__(
        self,
        model: nn.Module,
        tf_ids: torch.Tensor,
        tg_ids: torch.Tensor,
        bias: Optional[torch.Tensor] = None,
        motif_mask: Optional[torch.Tensor] = None,
        context_index: int = 0,
    ):
        super().__init__()
        self.model = model

        self.register_buffer("tf_ids_fixed", tf_ids.detach().clone())
        self.register_buffer("tg_ids_fixed", tg_ids.detach().clone())

        if bias is not None:
            if bias.dim() > 0 and bias.shape[0] > 1:
                self.register_buffer(
                    "bias_template",
                    bias[context_index:context_index + 1].detach().clone(),
                )
            else:
                self.register_buffer("bias_template", bias.detach().clone())
        else:
            self.bias_template = None

        if motif_mask is not None:
            if (
                bias is not None
                and motif_mask.dim() > 0
                and motif_mask.shape[0] > 1
                and motif_mask.shape[0] == bias.shape[0]
            ):
                self.register_buffer(
                    "motif_mask_template",
                    motif_mask[context_index:context_index + 1].detach().clone(),
                )
            else:
                self.register_buffer("motif_mask_template", motif_mask.detach().clone())
        else:
            self.motif_mask_template = None

    def _match_batch(self, x: Optional[torch.Tensor], batch_size: int):
        if x is None:
            return None
        if x.dim() == 1:
            return x
        if x.shape[0] == batch_size:
            return x
        if x.shape[0] == 1:
            reps = [batch_size] + [1] * (x.dim() - 1)
            return x.repeat(*reps)
        return x

    def forward(self, atac_wins: torch.Tensor, tf_tensor: Optional[torch.Tensor] = None):
        if tf_tensor is None:
            if isinstance(atac_wins, (list, tuple)) and len(atac_wins) == 2:
                atac_wins, tf_tensor = atac_wins
            else:
                raise ValueError(
                    "Expected forward(atac_wins, tf_tensor) or forward([atac_wins, tf_tensor])."
                )

        batch_size = atac_wins.shape[0]
        bias = self._match_batch(self.bias_template, batch_size)
        motif_mask = self._match_batch(self.motif_mask_template, batch_size)

        out, _, _ = self.model(
            atac_wins,
            tf_tensor,
            tf_ids=self.tf_ids_fixed,
            tg_ids=self.tg_ids_fixed,
            bias=bias,
            motif_mask=motif_mask,
            return_shortcut_contrib=False,
        )

        # shape: [batch, n_tgs]
        return out


def _to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _collect_background_same_context(
    atac_wins: torch.Tensor,
    tf_tensor: torch.Tensor,
    n_background: int = 16,
):
    n_background = min(int(n_background), atac_wins.shape[0])
    bg_atac = atac_wins[:n_background].detach().clone()
    bg_tf = tf_tensor[:n_background].detach().clone()
    return bg_atac, bg_tf


@torch.no_grad()
def predict_target_output(wrapper, atac_wins, tf_tensor):
    wrapper.eval()
    y = wrapper(atac_wins, tf_tensor)
    return _to_numpy(y)


def _unwrap_shap_values_single_output(shap_values):
    """
    For the single-output wrapper.
    Expected result is two attribution arrays: one for ATAC, one for TF.
    """
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            return shap_values[0], shap_values[1]
        if len(shap_values) == 1 and isinstance(shap_values[0], list) and len(shap_values[0]) == 2:
            return shap_values[0][0], shap_values[0][1]
    raise RuntimeError(f"Unexpected SHAP output structure: {type(shap_values)}")


def _unwrap_shap_values_multi_output(shap_values):
    """
    For the all-output wrapper.

    SHAP can return different nested formats depending on version and output shape.
    This function tries to normalize that into:
        phi_atac_np, phi_tf_np
    where each has output axis included somewhere.

    Common possibilities include:
      - list of length 2: [phi_atac, phi_tf]
      - list of outputs, each output containing [phi_atac, phi_tf]
      - other nested variants

    We normalize to numpy arrays and preserve all dimensions.
    """
    # Case 1: already input-wise list [atac, tf]
    if isinstance(shap_values, list) and len(shap_values) == 2 and not isinstance(shap_values[0], list):
        return _to_numpy(shap_values[0]), _to_numpy(shap_values[1])

    # Case 2: output-wise list: one entry per output, each entry like [phi_atac, phi_tf]
    if isinstance(shap_values, list) and len(shap_values) > 0 and isinstance(shap_values[0], list):
        phi_atac_list = []
        phi_tf_list = []
        for out_sv in shap_values:
            if not (isinstance(out_sv, list) and len(out_sv) == 2):
                raise RuntimeError(
                    f"Unexpected nested SHAP output structure inside multi-output list: {type(out_sv)}"
                )
            phi_atac_list.append(_to_numpy(out_sv[0]))
            phi_tf_list.append(_to_numpy(out_sv[1]))

        # Stack along output dimension at axis=0
        # Result shapes typically become:
        #   phi_atac: [n_outputs, batch, ...]
        #   phi_tf:   [n_outputs, batch, n_tfs]
        return np.stack(phi_atac_list, axis=0), np.stack(phi_tf_list, axis=0)

    raise RuntimeError(f"Unexpected SHAP output structure for multi-output case: {type(shap_values)}")


def _extract_tf_scores_per_output(phi_tf, n_outputs: int):
    """
    Convert multi-output TF SHAP output into a clean array of shape:
        [n_outputs, n_tfs]

    Handles a couple common layouts:
      1) [n_outputs, batch, n_tfs]
      2) [batch, n_tfs, n_outputs]
      3) [batch, n_outputs, n_tfs]
      4) [n_outputs, batch, ..., n_tfs] -> flattened over non-batch, non-output dims
    """
    arr = np.abs(_to_numpy(phi_tf))

    if arr.ndim < 2:
        raise RuntimeError(f"phi_tf has too few dims: shape={arr.shape}")

    # Case: [n_outputs, batch, n_tfs]
    if arr.ndim == 3 and arr.shape[0] == n_outputs:
        # use first explained sample
        return arr[:, 0, :]

    # Case: [batch, n_tfs, n_outputs]
    if arr.ndim == 3 and arr.shape[-1] == n_outputs:
        return arr[0, :, :].T

    # Case: [batch, n_outputs, n_tfs]
    if arr.ndim == 3 and arr.shape[1] == n_outputs:
        return arr[0, :, :]

    # More general fallback:
    # find output axis by matching n_outputs
    candidate_axes = [ax for ax, size in enumerate(arr.shape) if size == n_outputs]
    if not candidate_axes:
        raise RuntimeError(
            f"Could not identify output axis in phi_tf with shape={arr.shape} and n_outputs={n_outputs}"
        )

    output_axis = candidate_axes[0]
    arr = np.moveaxis(arr, output_axis, 0)  # [n_outputs, ...]

    # Assume first remaining axis is batch if present; take first sample
    if arr.ndim >= 3:
        arr = arr[:, 0, ...]

    # Flatten everything except output axis
    arr = arr.reshape(n_outputs, -1)
    return arr


def test_shap_input_output_single_sample(
    exp,
    checkpoint_path="trained_model.pt",
    sample_index=0,
    target_tg_idx=0,
    reduce="single",
    n_background=16,
    shap_nsamples=128,
    batch_index=0,
):
    """
    Original single-target debugging function retained for validation.
    """
    exp.load_trained_model(checkpoint_path)
    exp.model.eval()

    device = exp.device
    loader_iter = iter(exp.test_loader)
    batch = None
    for _ in range(batch_index + 1):
        batch = next(loader_iter)

    atac_wins, tf_tensor, tg_expr_true, bias, tf_ids, tg_ids, motif_mask = batch

    atac_wins = atac_wins.to(device, non_blocking=True)
    tf_tensor = tf_tensor.to(device, non_blocking=True)
    tg_expr_true = tg_expr_true.to(device, non_blocking=True)
    bias = bias.to(device, non_blocking=True) if bias is not None else None
    tf_ids = tf_ids.to(device, non_blocking=True)
    tg_ids = tg_ids.to(device, non_blocking=True)
    motif_mask = motif_mask.to(device, non_blocking=True) if motif_mask is not None else None

    bg_atac, bg_tf = _collect_background_same_context(
        atac_wins=atac_wins,
        tf_tensor=tf_tensor,
        n_background=n_background,
    )

    x_atac = atac_wins[sample_index:sample_index + 1].clone().detach().requires_grad_(True)
    x_tf = tf_tensor[sample_index:sample_index + 1].clone().detach().requires_grad_(True)

    wrapped_model = TGOutputWrapperSingleContext(
        model=exp.model,
        tf_ids=tf_ids,
        tg_ids=tg_ids,
        bias=bias,
        motif_mask=motif_mask,
        target_tg_idx=target_tg_idx,
        reduce=reduce,
        context_index=sample_index,
    ).to(device)
    wrapped_model.eval()

    explainer = shap.GradientExplainer(
        wrapped_model,
        data=[bg_atac, bg_tf],
    )

    shap_values = explainer.shap_values(
        [x_atac, x_tf],
        nsamples=shap_nsamples,
    )

    phi_atac, phi_tf = _unwrap_shap_values_single_output(shap_values)

    y_pred = predict_target_output(wrapped_model, x_atac, x_tf)
    base_value = predict_target_output(wrapped_model, bg_atac, bg_tf).mean()

    phi_atac_np = _to_numpy(phi_atac)
    phi_tf_np = _to_numpy(phi_tf)

    recon = (
        base_value
        + phi_atac_np.reshape(phi_atac_np.shape[0], -1).sum(axis=1)
        + phi_tf_np.reshape(phi_tf_np.shape[0], -1).sum(axis=1)
    )

    return {
        "sample_index": sample_index,
        "target_tg_idx": target_tg_idx,
        "reduce": reduce,
        "base_value": float(base_value),
        "prediction": y_pred,
        "reconstructed_prediction": recon,
        "reconstruction_error": y_pred - recon,
        "shap_atac": phi_atac_np,
        "shap_tf": phi_tf_np,
        "mean_abs_shap_atac": np.abs(phi_atac_np[0]).reshape(-1),
        "mean_abs_shap_tf": np.abs(phi_tf_np[0]).reshape(-1),
        "explained_atac": _to_numpy(x_atac),
        "explained_tf": _to_numpy(x_tf),
        "true_tg_expr_sample": _to_numpy(tg_expr_true[sample_index:sample_index + 1]),
        "tf_ids_sample": _to_numpy(tf_ids[sample_index:sample_index + 1]),
        "tg_ids_sample": _to_numpy(tg_ids[sample_index:sample_index + 1]),
    }


def build_df_wide_shap(
    exp,
    checkpoint_path="trained_model.pt",
    sample_index=0,
    batch_indices=None,
    n_background=8,
    shap_nsamples=32,
    verbose=False,
):
    """
    Faster version:
      - builds ONE GradientExplainer per batch
      - explains ALL TG outputs at once
      - aggregates mean(|SHAP_tf|) per TG across selected batches

    Returns
    -------
    df_wide : pd.DataFrame
        Rows = TFs
        Columns = TGs
        Values = mean absolute TF SHAP scores
    """
    exp.load_trained_model(checkpoint_path)
    exp.model.eval()

    if batch_indices is None:
        batch_indices = list(range(len(exp.test_loader)))
    else:
        batch_indices = list(batch_indices)

    batch_set = set(batch_indices)

    agg_sum = {}
    agg_count = {}

    for b_idx, batch in enumerate(exp.test_loader):
        if b_idx not in batch_set:
            continue

        atac_wins, tf_tensor, tg_expr_true, bias, tf_ids, tg_ids, motif_mask = batch

        atac_wins = atac_wins.to(exp.device, non_blocking=True)
        tf_tensor = tf_tensor.to(exp.device, non_blocking=True)
        tg_expr_true = tg_expr_true.to(exp.device, non_blocking=True)
        bias = bias.to(exp.device, non_blocking=True) if bias is not None else None
        tf_ids = tf_ids.to(exp.device, non_blocking=True)
        tg_ids = tg_ids.to(exp.device, non_blocking=True)
        motif_mask = motif_mask.to(exp.device, non_blocking=True) if motif_mask is not None else None

        if sample_index >= atac_wins.shape[0]:
            raise IndexError(
                f"sample_index={sample_index} is out of range for batch {b_idx} "
                f"with batch size {atac_wins.shape[0]}"
            )

        if tg_ids.dim() == 1:
            tg_globals = tg_ids
        else:
            tg_globals = tg_ids[0]

        n_outputs = int(len(tg_globals))

        bg_atac, bg_tf = _collect_background_same_context(
            atac_wins=atac_wins,
            tf_tensor=tf_tensor,
            n_background=n_background,
        )

        x_atac = atac_wins[sample_index:sample_index + 1].clone().detach().requires_grad_(True)
        x_tf = tf_tensor[sample_index:sample_index + 1].clone().detach().requires_grad_(True)

        wrapped_model = TGOutputWrapperAll(
            model=exp.model,
            tf_ids=tf_ids,
            tg_ids=tg_ids,
            bias=bias,
            motif_mask=motif_mask,
            context_index=sample_index,
        ).to(exp.device)
        wrapped_model.eval()

        explainer = shap.GradientExplainer(
            wrapped_model,
            data=[bg_atac, bg_tf],
        )

        shap_values = explainer.shap_values(
            [x_atac, x_tf],
            nsamples=shap_nsamples,
        )

        phi_atac, phi_tf = _unwrap_shap_values_multi_output(shap_values)
        tf_scores_by_output = _extract_tf_scores_per_output(phi_tf, n_outputs=n_outputs)

        if verbose:
            print(f"Batch {b_idx}")
            print("  tg count:", n_outputs)
            print("  phi_atac shape:", np.shape(phi_atac))
            print("  phi_tf shape:", np.shape(phi_tf))
            print("  tf_scores_by_output shape:", np.shape(tf_scores_by_output))

        for local_idx, tg_global in enumerate(tg_globals.tolist()):
            tf_scores = tf_scores_by_output[local_idx].reshape(-1)

            if tg_global not in agg_sum:
                agg_sum[tg_global] = tf_scores.copy()
                agg_count[tg_global] = 1
            else:
                agg_sum[tg_global] += tf_scores
                agg_count[tg_global] += 1

    if len(agg_sum) == 0:
        raise RuntimeError("No TG scores were aggregated. Check batch_indices and SHAP output structure.")

    cols = []
    data = []
    for tg_global in sorted(agg_sum.keys()):
        cols.append(exp.tg_names[int(tg_global)])
        data.append(agg_sum[tg_global] / max(agg_count[tg_global], 1))

    df_wide = pd.DataFrame(
        np.vstack(data).T,
        index=exp.tf_names,
        columns=cols,
    )
    return df_wide


# Example usage:
df_wide = build_df_wide_shap(
    exp,
    checkpoint_path="trained_model.pt",
    sample_index=0,
    batch_indices=[0, 1],
    n_background=8,
    shap_nsamples=32,
    verbose=True,
)

df_wide.head()

In [122]:
from scipy.stats import norm

def wide_to_score_df(df_wide):
    df = (
        df_wide
        .reset_index(names="Source")
        .melt(id_vars="Source", var_name="Target", value_name="Score")
    )
    
    def inverse_normal_transform(x):
        r = x.rank(method="average")
        n = len(x)
        p = (r - 0.5) / n          # avoids 0 and 1
        return norm.ppf(p)
    
    # Apply rank-based inverse normal transform (INT)
    df["Score"] = df.groupby("Source")["Score"].transform(inverse_normal_transform)
    
    df = df.dropna()
    
    df["Source"] = df["Source"].astype(str).str.upper()
    df["Target"] = df["Target"].astype(str).str.upper()

    return df

In [127]:
ground_truth_name = "ChIP-Atlas mESC"
ground_truth = gt_by_dataset_dict["mESC"][ground_truth_name]

score_df = wide_to_score_df(df_wide)
print(score_df.head())

# pooled_metrics_df_flipped = exp.generate_pooled_metrics(
#     method_name="Gradient Attribution",
#     score_df=score_df,
#     ground_truth=ground_truth,
#     ground_truth_name=ground_truth_name,
#     balance=True,
# )

# fig = exp.plot_hist_roc_pr(
#     score_df,
#     gt_by_dataset_dict["mESC"],
#     method_name="Gradient Attribution",
#     y_log=False,
#     panel_kind="hist"
# )
# fig.show()

per_tf_auroc_all_gt = []
pooled_tf_auroc_all_gt = []
print(f"----- GT Overlap -----")
for gt_name, gt_df in gt_by_dataset_dict["mESC"].items():
    
    labeled_df = exp.create_ground_truth_comparison_df(score_df, gt_df[1], ground_truth_name)

    # --- Micro (pooled)
    y_all = labeled_df["_in_gt"].astype(int).to_numpy()
    overlap_df = labeled_df[labeled_df["_in_gt"] == 1]
    overlapping_tfs = overlap_df['Source'].nunique()
    overlapping_tgs = overlap_df['Target'].nunique()
    edge_overlap = overlap_df.shape[0]
    print(f"  - {gt_name} Overlapping TFs: {overlapping_tfs}, TGs: {overlapping_tgs}, edges: {edge_overlap}")

    s_all = labeled_df["Score"].to_numpy()
    micro_auroc = roc_auc_score(y_all, s_all)

    # --- Macro (median per TF)
    per_tf = []
    for tf, sub in labeled_df.groupby("Source"):
        if sub["_in_gt"].nunique() < 2:
            continue
        y = sub["_in_gt"].astype(int).to_numpy()
        s = sub["Score"].to_numpy()
        per_tf.append(roc_auc_score(y, s))

    macro_auroc = np.median(per_tf)
    
    per_tf_auroc_all_gt.append(macro_auroc)
    pooled_tf_auroc_all_gt.append(micro_auroc)

print(f"\n----- AUROC -----")
print(f"Median pooled AUROC: {np.median(pooled_tf_auroc_all_gt):.3f}")
print(f"Median per-TF AUROC: {np.median(per_tf_auroc_all_gt):.3f}")

   Source Target     Score
0    ADNP   AAMP  0.354976
1   ADNP2   AAMP -0.013907
2    AFDN   AAMP  0.768078
3  AHCTF1   AAMP  1.225128
4     AHR   AAMP -0.559839
----- GT Overlap -----
  - ChIP-Atlas mESC Overlapping TFs: 40, TGs: 709, edges: 7783
  - RN111 Overlapping TFs: 43, TGs: 653, edges: 8633
  - RN112 Overlapping TFs: 19, TGs: 377, edges: 936
  - RN114 Overlapping TFs: 16, TGs: 519, edges: 1821
  - RN116 Overlapping TFs: 9, TGs: 122, edges: 158

----- AUROC -----
Median pooled AUROC: 0.516
Median per-TF AUROC: 0.517
